In [8]:
import numpy as np
import torch
import itertools

import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from pyro.infer import MCMC, NUTS, HMC, SVI, Trace_ELBO, Predictive
from pyro.optim import Adam, ClippedAdam

In [9]:
# Start by loading the data
# We only need y because we will not be looking at any features

y_train = np.load("../../data/processed/y_train.npy")
y_test = np.load("../../data/processed/y_test.npy")

y_train_torch = torch.tensor(y_train, dtype=torch.float32)
y_test_np = y_test.astype(float)


In [5]:
# Define the baseline model
def constant_poisson_model(y=None):
    n = len(y)
    log_rate = pyro.sample("log_rate", dist.Normal(0., 1.))
    rate = torch.exp(log_rate)
    
    with pyro.plate("data", n):
        pyro.sample("obs", dist.Poisson(rate), obs=y)

In [6]:
# Define params
n_steps = 3000
lr = 0.005

In [7]:
# Now we train the model
pyro.clear_param_store()

guide = AutoDiagonalNormal(constant_poisson_model)

optimizer = ClippedAdam({"lr": lr})
svi = SVI(constant_poisson_model, guide, optimizer, loss=Trace_ELBO())

for step in range(n_steps):
    loss = svi.step(y_train_torch)
    
    if step % 500 == 0:
        print(f"[{step}] ELBO loss: {loss:.2f}")

[0] ELBO loss: 755014.92
[500] ELBO loss: 513661.42
[1000] ELBO loss: 513187.44
[1500] ELBO loss: 513199.63
[2000] ELBO loss: 513191.01
[2500] ELBO loss: 513189.05


In [ ]:
num_samples = 1000
n_test = len(y_test_np)

predictive = Predictive(
    constant_poisson_model,
    guide=guide,
    num_samples=num_samples,
    return_sites=("log_rate",)
)

dummy_y = torch.zeros(n_test)
samples = predictive(dummy_y)

rate_samples = torch.exp(samples["log_rate"]).squeeze()

mean_rate = rate_samples.mean().item()
y_pred = np.full(n_test, mean_rate)

In [ ]:
# Define function to compute error
def compute_error(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / denominator

    return mae, rmse, r2

In [ ]:
mae, rmse, r2 = compute_error(y_test_np, y_pred)

print("\nBaseline Poisson model")
print(f"Estimated constant rate prediction: {preds[0]:.3f}")
print(f"CorrCoef: undefined for constant baseline")
print(f"MAE: {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R2: {r2:.3f}")


num train: 204615
num test: 105408
